In [2]:
!pip install "imitation>=1.0.0" "stable-baselines3[extra]>=2.0.0" "gymnasium>=0.28.1"

import gymnasium as gym
import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy

from imitation.algorithms.adversarial.gail import GAIL
from imitation.data import rollout
from imitation.rewards.reward_nets import BasicRewardNet
from imitation.util.networks import RunningNorm
from imitation.util.util import make_vec_env

# Configurar entorno y política experta
rng = np.random.default_rng(0)
venv = make_vec_env("CartPole-v1", n_envs=8, rng=rng)

print("Entrenando al experto en CartPole-v1...")
expert = PPO("MlpPolicy", venv, verbose=0)
expert.learn(10000)
print("Experto entrenado.")

mean_reward_expert, _ = evaluate_policy(expert, venv, n_eval_episodes=10, deterministic=True)
print(f"Recompensa promedio del experto: {mean_reward_expert:.2f}")

# Generar demostraciones del experto
print("\nGenerando demostraciones del experto...")
expert_transitions = rollout.rollout(
    expert,
    venv,
    rollout.make_sample_until(min_timesteps=None, min_episodes=50),
    rng=rng,
    unwrap=False,
)
print("Demostraciones generadas.")

# Crear el agente que aprenderá por imitación
learner = PPO("MlpPolicy", venv, verbose=1)

# Crear la red de recompensa (discriminador)
print("\nCreando la red de recompensa (discriminador)...")
reward_net = BasicRewardNet(
    observation_space=venv.observation_space,
    action_space=venv.action_space,
    normalize_input_layer=RunningNorm,
)
print("Red de recompensa creada.")

# Entrenar con GAIL
gail_trainer = GAIL(
    demonstrations=expert_transitions,
    demo_batch_size=1024,
    gen_algo=learner,
    venv=venv,
    reward_net=reward_net,
    n_disc_updates_per_round=4,
    allow_variable_horizon=True,
)

print("\nEntrenando al agente aprendiz con GAIL...")
gail_trainer.train(total_timesteps=20000)

# Evaluar la política aprendida por imitación
print("\nEvaluando la política final aprendida por imitación...")
mean_reward_learner, _ = evaluate_policy(learner, venv, n_eval_episodes=10, deterministic=True)

print(f"\n===== RESULTADO FINAL =====")
print(f"Recompensa promedio del experto: {mean_reward_expert:.2f}")
print(f"Recompensa promedio de la política aprendida con GAIL: {mean_reward_learner:.2f}")

INFO: pip is looking at multiple versions of stable-baselines3[extra] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 7.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.5/216.5 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.7/181.7 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 953.9/953.9 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.2/108.2 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 81

Entrenando al experto en CartPole-v1...
Experto entrenado.
Recompensa promedio del experto: 431.80

Generando demostraciones del experto...
Demostraciones generadas.
Using cpu device

Creando la red de recompensa (discriminador)...
Red de recompensa creada.
Running with `allow_variable_horizon` set to True. Some algorithms are biased towards shorter or longer episodes, which may significantly confound results. Additionally, even unbiased algorithms can exploit the information leak from the termination condition, producing spuriously high performance. See https://imitation.readthedocs.io/en/latest/getting-started/variable-horizon.html for more information.

Entrenando al agente aprendiz con GAIL...


round:   0%|          | 0/1 [00:00<?, ?it/s]

------------------------------------------
| raw/                        |          |
|    gen/rollout/ep_len_mean  | 23.1     |
|    gen/rollout/ep_rew_mean  | 23.1     |
|    gen/time/fps             | 4756     |
|    gen/time/iterations      | 1        |
|    gen/time/time_elapsed    | 3        |
|    gen/time/total_timesteps | 16384    |
------------------------------------------
--------------------------------------------------
| raw/                                |          |
|    disc/disc_acc                    | 0.513    |
|    disc/disc_acc_expert             | 0.49     |
|    disc/disc_acc_gen                | 0.535    |
|    disc/disc_entropy                | 0.693    |
|    disc/disc_loss                   | 0.695    |
|    disc/disc_proportion_expert_pred | 0.478    |
|    disc/disc_proportion_expert_true | 0.5      |
|    disc/global_step                 | 1        |
|    disc/n_expert                    | 1.02e+03 |
|    disc/n_generated                 | 1.02e+03 |
-

round: 100%|██████████| 1/1 [00:13<00:00, 13.27s/it]


Evaluando la política final aprendida por imitación...



===== RESULTADO FINAL =====
Recompensa promedio del experto: 431.80
Recompensa promedio de la política aprendida con GAIL: 352.80
